In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_features_sales_timeseries = get_superstore_logger("superstore_features_sales_timeseries")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_features_sales_timeseries,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_features_sales_timeseries, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create feature saless timeseries table.", layer="Feature Engineering")
else:
    # -------------------------------
    # Create or update features_sales_timeseries Table (Environment-aware)
    # -------------------------------
    # Check if the sales timeseries features table already exists
    if spark.catalog.tableExists(f"{catalog}.{features_schema}.features_sales_timeseries"):
        # If the table exists, perform MERGE to update existing rows and insert new ones
        spark.sql(f"""
            MERGE INTO {catalog}.{features_schema}.features_sales_timeseries AS target
            USING (
                -- 1. Base sales data per product per day
                WITH base_sales AS (
                    SELECT
                        o.order_date,
                        s.product_id,
                        s.sales AS total_sales
                    FROM {catalog}.{gold_schema}.facts_sales s
                    JOIN {catalog}.{gold_schema}.facts_orders o
                        ON s.order_id = o.order_id
                ),

                -- 2. Generate time-series features
                time_features AS (
                    SELECT
                        order_date,
                        product_id,

                        -- Lag features to capture previous sales without leakage
                        LAG(total_sales, 7) OVER (PARTITION BY product_id ORDER BY order_date) AS lag_7_sales,
                        LAG(total_sales, 30) OVER (PARTITION BY product_id ORDER BY order_date) AS lag_30_sales,

                        -- Rolling averages (exclude current row)
                        AVG(total_sales) OVER (
                            PARTITION BY product_id
                            ORDER BY order_date
                            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
                        ) AS rolling_avg_7,
                        AVG(total_sales) OVER (
                            PARTITION BY product_id
                            ORDER BY order_date
                            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
                        ) AS rolling_avg_30,

                        -- Calendar features
                        DAYOFWEEK(order_date) AS day_of_week,
                        MONTH(order_date) AS month,

                        ROW_NUMBER() OVER (PARTITION BY product_id, order_date ORDER BY order_date DESC) AS row_num  -- De-duplication
                    FROM base_sales
                )
                -- Only keep the first row (ROW_NUMBER = 1) for each product_id and order_date pair
                SELECT * FROM time_features WHERE row_num = 1
            ) AS source
            ON target.product_id = source.product_id
            AND target.order_date = source.order_date

            -- Update existing records if matched
            WHEN MATCHED THEN
                UPDATE SET
                    target.lag_7_sales = source.lag_7_sales,
                    target.lag_30_sales = source.lag_30_sales,
                    target.rolling_avg_7 = source.rolling_avg_7,
                    target.rolling_avg_30 = source.rolling_avg_30,
                    target.day_of_week = source.day_of_week,
                    target.month = source.month

            -- Insert new records if not matched
            WHEN NOT MATCHED THEN
                INSERT (order_date, product_id, lag_7_sales, lag_30_sales, rolling_avg_7, rolling_avg_30, day_of_week, month)
                VALUES (source.order_date, source.product_id, source.lag_7_sales, source.lag_30_sales, source.rolling_avg_7, source.rolling_avg_30, source.day_of_week, source.month)
        """)
    else:
        # Table does not exist, create it from scratch
        spark.sql(f"""
            -- Create the Sales Time-Series ML Features Table
            CREATE TABLE {catalog}.{features_schema}.features_sales_timeseries AS
            WITH base_sales AS (
                SELECT
                    o.order_date,
                    s.product_id,
                    s.sales AS total_sales
                FROM {catalog}.{gold_schema}.facts_sales s
                JOIN {catalog}.{gold_schema}.facts_orders o
                    ON s.order_id = o.order_id
            ),
            time_features AS (
                SELECT
                    order_date,
                    product_id,
                    LAG(total_sales, 7) OVER (PARTITION BY product_id ORDER BY order_date) AS lag_7_sales,
                    LAG(total_sales, 30) OVER (PARTITION BY product_id ORDER BY order_date) AS lag_30_sales,
                    AVG(total_sales) OVER (
                        PARTITION BY product_id
                        ORDER BY order_date
                        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
                    ) AS rolling_avg_7,
                    AVG(total_sales) OVER (
                        PARTITION BY product_id
                        ORDER BY order_date
                        ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
                    ) AS rolling_avg_30,
                    DAYOFWEEK(order_date) AS day_of_week,
                    MONTH(order_date) AS month,
                    ROW_NUMBER() OVER (PARTITION BY product_id, order_date ORDER BY order_date DESC) AS row_num  -- De-duplication
                FROM base_sales
            )
            SELECT * FROM time_features WHERE row_num = 1
        """)
log_event(logger_features_sales_timeseries, "INFO", "Feature customer table created successfully.", layer="Feature Engineering")
